[1] 모듈로딩 및 데이터 준비 <hr>

In [1]:
## [1-1] 모듈 로딩
import pandas as pd
import numpy as np

from ultralytics import YOLO
from ultralytics import RTDETR 

import cv2
from matplotlib import pyplot as plt

In [ ]:
# ## [1-2] 데이터 준비
# import yt_dlp

# url = "https://www.youtube.com/watch?v=WVb-cQUID2A"

# ydl_opts = {"format": "best"}

# with yt_dlp.YoutubeDL(ydl_opts) as ydl:
#     info = ydl.extract_info(url, download=False) 
#     stream_url = info["url"]

# cap = cv2.VideoCapture(stream_url)

In [2]:
## ===============================================
## [2-2] IoU 계산 함수
## ===============================================
def bbox_iou(box1, box2):
    xA = max(box1[0], box2[0])
    yA = max(box1[1], box2[1])
    xB = min(box1[2], box2[2])
    yB = min(box1[3], box2[3])

    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)
    inter_area = inter_w * inter_h

    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])

    union = area1 + area2 - inter_area
    if union == 0:
        return 0

    return inter_area / union

In [3]:
## ===============================================
## [2-3] Occlusion 판단 함수 (track 기반)
## ===============================================
def detect_occlusion_tracks(tracks, threshold=0.3):
    occluded_ids = set()

    for i in range(len(tracks)):
        for j in range(len(tracks)):
            if i == j:
                continue

            box_i = tracks[i]["box"]
            box_j = tracks[j]["box"]

            # intersection 계산
            xA = max(box_i[0], box_j[0])
            yA = max(box_i[1], box_j[1])
            xB = min(box_i[2], box_j[2])
            yB = min(box_i[3], box_j[3])

            inter_w = max(0, xB - xA)
            inter_h = max(0, yB - yA)
            inter_area = inter_w * inter_h

            area_i = (box_i[2]-box_i[0])*(box_i[3]-box_i[1])
            area_j = (box_j[2]-box_j[0])*(box_j[3]-box_j[1])

            if inter_area == 0:
                continue

            # 🔥 핵심 부분
            occlusion_ratio = inter_area / min(area_i, area_j)

            if occlusion_ratio > threshold:
                if area_i < area_j:
                    occluded_ids.add(tracks[i]["id"])
                else:
                    occluded_ids.add(tracks[j]["id"])

    return occluded_ids

In [4]:
## ===============================================
## [2-4] occlusion_1 영상 처리 루프
## ===============================================
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # ByteTrack 포함 추적 실행
    results = model.track(frame,
                          persist=True,
                          tracker=tracker,
                          conf=0.4,
                          verbose=False)

    tracks = []

    if results[0].boxes.id is not None:

        for box, track_id, cls in zip(results[0].boxes.xyxy,
                                      results[0].boxes.id,
                                      results[0].boxes.cls):

            cls_id = int(cls)

            # 차량 클래스만 (COCO 기준)
            if cls_id in [2, 5, 7]:
                x1, y1, x2, y2 = map(int, box)
                tracks.append({
                    "id": int(track_id),
                    "box": [x1, y1, x2, y2]
                })

    ## -----------------------------------------------------
    # Occlusion 판단
    ## -----------------------------------------------------
    ## 1) bbox 겹칠 경우
    occluded_ids = detect_occlusion_tracks(tracks, threshold=0.1)

    ## 2) 중앙분리대에 가린 경우

    ## -----------------------------------------------------
    # 박스 그리기
    ## -----------------------------------------------------
    for track in tracks:
        x1, y1, x2, y2 = track["box"]
        track_id = track["id"]

        if track_id in occluded_ids:
            color = (0, 0, 255)  ##빨강
            label = f"ID {track_id} - Occlusion"
        else:
            color = (0, 255, 0)  ##초록
            label = f"ID {track_id}"

        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, label, (x1, y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, color, 2)

    cv2.imshow("YOLOv26 + ByteTrack Occlusion", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

NameError: name 'cap' is not defined

In [ ]:
# ## ===============================================
# ## [2-4] occlusion_2 영상 처리 루프
# ## ===============================================
# while cap.isOpened():
#     ret, frame = cap.read()
#     if not ret:
#         break

#     results = model.track(frame,
#                           persist=True,
#                           tracker="bytetrack.yaml",
#                           conf=0.4,
#                           verbose=False)

#     if results[0].boxes.id is None or results[0].masks is None:
#         cv2.imshow("Seg Occlusion", frame)
#         if cv2.waitKey(1) & 0xFF == 27:
#             break
#         continue

#     boxes = results[0].boxes.xyxy.cpu().numpy()
#     ids = results[0].boxes.id.cpu().numpy()
#     classes = results[0].boxes.cls.cpu().numpy()
#     masks = results[0].masks.data.cpu().numpy()

#     for box, track_id, cls_id, mask in zip(boxes, ids, classes, masks):

#         # 차량 클래스만 (COCO 기준)
#         if int(cls_id) not in [2, 5, 7, 9]:
#             continue

#         x1, y1, x2, y2 = map(int, box)

#         # -----------------------------
#         # bbox 정보 계산
#         # -----------------------------
#         w = x2 - x1
#         h = y2 - y1

#         if w <= 0 or h <= 0:
#             continue

#         ratio = w / h
#         bbox_area = w * h

#         # -----------------------------
#         # mask 기반 visible ratio 계산
#         # -----------------------------
#         mask_binary = mask.astype(np.uint8)

#         # bbox 내부 mask만 사용
#         mask_crop = mask_binary[y1:y2, x1:x2]

#         visible_area = np.sum(mask_crop)

#         if bbox_area == 0:
#             continue

#         visible_ratio = visible_area / bbox_area

#         # -----------------------------
#         # 🔥 Occlusion 조건 (2단계)
#         # -----------------------------
#         occlusion_flag = False

#         # 조건 1: mask 비율
#         if visible_ratio < 0.45:
#             occlusion_flag = True

#         # 조건 2: 비정상적으로 납작한 박스
#         if ratio > 2.0:
#             occlusion_flag = True

#         # -----------------------------
#         # 박스 그리기
#         # -----------------------------
#         if occlusion_flag:
#             color = (0, 0, 255)  ## 빨강
#             label = f"ID {int(track_id)} - Occlusion"
#         else:
#             color = (0, 255, 0)  ## 초록
#             label = f"ID {int(track_id)}"

#         cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
#         cv2.putText(frame,
#                     f"{label} | VR:{visible_ratio:.2f}",
#                     (x1, y1 - 5),
#                     cv2.FONT_HERSHEY_SIMPLEX,
#                     0.5, color, 2)

#     cv2.imshow("Seg Occlusion", frame)

#     if cv2.waitKey(1) & 0xFF == 27:
#         break

# cap.release()
# cv2.destroyAllWindows()

In [ ]:
# while cap.isOpened():
#     ret, frame = cap.read()
#     if not ret:
#         break

#     results = model.track(frame,
#                           persist=True,
#                           tracker= tracker,
#                           conf=0.4,
#                           verbose=False)

#     if results[0].boxes.id is None:
#         cv2.imshow("Seg Occlusion", frame)
#         if cv2.waitKey(1) & 0xFF == 27:
#             break
#         continue

#     boxes = results[0].boxes.xyxy.cpu().numpy()
#     ids = results[0].boxes.id.cpu().numpy()
#     classes = results[0].boxes.cls.cpu().numpy()

#     for box, track_id, cls_id in zip(boxes, ids, classes):

#         # 차량 클래스만
#         if int(cls_id) not in [2, 5, 7, 9]:
#             continue

#         x1, y1, x2, y2 = map(int, box)

#         # -----------------------------
#         # 🔥 중앙분리대 Occlusion 판단
#         # -----------------------------
#         occlusion_flag = False

#         frame_h = frame.shape[0]
#         bottom_ratio = y2 / frame_h

#         # bbox 영역 crop
#         roi = frame[y1:y2, x1:x2]

#         if roi.shape[0] < 20:
#             continue

#         # 하단 25% 영역만 검사
#         h = roi.shape[0]
#         bottom_roi = roi[int(h * 0.75):h, :]

#         hsv = cv2.cvtColor(bottom_roi, cv2.COLOR_BGR2HSV)

#         # -----------------------------
#         # 1️⃣ 회색 콘크리트 검출
#         # -----------------------------
#         gray_mask = cv2.inRange(
#             hsv,
#             (0, 0, 50),
#             (180, 40, 200)
#         )
#         gray_ratio = np.mean(gray_mask) / 255

#         # -----------------------------
#         # 2️⃣ 노랑 검출
#         # -----------------------------
#         yellow_mask = cv2.inRange(
#             hsv,
#             (20, 80, 80),
#             (35, 255, 255)
#         )
#         yellow_ratio = np.mean(yellow_mask) / 255

#         # -----------------------------
#         # 3️⃣ 검정 검출
#         # -----------------------------
#         black_mask = cv2.inRange(
#             hsv,
#             (0, 0, 0),
#             (180, 255, 60)
#         )
#         black_ratio = np.mean(black_mask) / 255

#         pattern_flag = yellow_ratio > 0.12 and black_ratio > 0.12

#         # -----------------------------
#         # 최종 판단
#         # -----------------------------
#         if bottom_ratio > 0.75:
#             if gray_ratio > 0.35 or pattern_flag:
#                 occlusion_flag = True

#         # -----------------------------
#         # 박스 그리기
#         # -----------------------------
#         if occlusion_flag:
#             color = (0, 0, 255)
#             label = f"ID {int(track_id)} - Occlusion"
#         else:
#             color = (0, 255, 0)
#             label = f"ID {int(track_id)}"

#         cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
#         cv2.putText(frame, label,
#                     (x1, y1 - 5),
#                     cv2.FONT_HERSHEY_SIMPLEX,
#                     0.5, color, 2)

#     cv2.imshow("Seg Occlusion", frame)

#     if cv2.waitKey(1) & 0xFF == 27:
#         break

# cap.release()
# cv2.destroyAllWindows()

In [5]:
import cv2
import os
from ultralytics import RTDETR

def main():
    # 1. 모델 및 비디오 설정
    model = RTDETR("rtdetr-l.pt")
    
    # 💡 파일 경로들을 확인해줘!
    video_path = "./dataset/source_videos/segment_011.mp4" 
    save_dir = "./dataset/output_crops"
    
    # 저장 폴더가 없으면 생성
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
        print(f"폴더 생성 완료: {save_dir}")

    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # 왼쪽 40% 기준선
    left_40_percent = int(width * 0.4) 
    
    frame_count = 0
    saved_count = 0

    print("데이터 수집을 시작합니다. 잠시만 기다려주세요...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        frame_count += 1

        # 2. 모델 트래킹 실행 (디스플레이가 없으므로 verbose=False 권장)
        results = model.track(frame, persist=True, tracker="bytetrack.yaml", conf=0.4, verbose=False)

        if results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            ids = results[0].boxes.id.cpu().numpy()
            classes = results[0].boxes.cls.cpu().numpy()

            for box, track_id, cls_id in zip(boxes, ids, classes):
                cls_id = int(cls_id)
                track_id = int(track_id)
                
                # 자동차(2), 버스(5), 트럭(7)만 필터링
                if cls_id not in [2, 5, 7]:
                    continue

                x1, y1, x2, y2 = map(int, box)
                cx = (x1 + x2) // 2
                
                # 3. 왼쪽 40% 구역 안에 있는 경우만 처리
                if cx <= left_40_percent:
                    box_h = y2 - y1
                    
                    # 클래스별 확장 비율 적용
                    if cls_id == 2:
                        extended_y2 = int(y1 + (box_h * 4))
                        cls_name = "car"
                    else:
                        extended_y2 = int(y1 + (box_h * 1.8))
                        cls_name = "heavy"
                    
                    # 화면 경계 제한
                    extended_y2 = min(extended_y2, height)
                    
                    # ---------------------------------------------------------
                    # 4. 이미지 크롭 및 저장
                    # ---------------------------------------------------------
                    # 가끔 박스가 화면 밖으로 나가는 경우를 대비해 좌표 보정
                    x1, y1 = max(0, x1), max(0, y1)
                    x2, extended_y2 = min(width, x2), min(height, extended_y2)
                    
                    # 크롭 실행
                    crop_img = frame[y1:extended_y2, x1:x2]
                    
                    # 이미지가 정상적으로 잘렸을 때만 저장
                    if crop_img.size > 0:
                        file_name = f"{cls_name}_id{track_id}_f{frame_count}.jpg"
                        save_path = os.path.join(save_dir, file_name)
                        cv2.imwrite(save_path, crop_img)
                        saved_count += 1

        # 진행 상황 출력 (100프레임마다)
        if frame_count % 100 == 0:
            print(f"진행 중... {frame_count} 프레임 처리 완료 (저장된 이미지: {saved_count}장)")

    cap.release()
    print("-" * 30)
    print(f"수집 완료!")
    print(f"총 처리 프레임: {frame_count}")
    print(f"총 저장 이미지: {saved_count}장")
    print(f"저장 위치: {os.path.abspath(save_dir)}")

if __name__ == "__main__":
    main()

데이터 수집을 시작합니다. 잠시만 기다려주세요...
------------------------------
수집 완료!
총 처리 프레임: 0
총 저장 이미지: 0장
저장 위치: c:\Users\choju\Desktop\driving-beam\AI\dataset\output_crops
